# drawdown_rebound_checkpoint_pbo_v1

Notebook-owned checkpoint/model-selection diagnostic after `drawdown_rebound_reward_train_v1` failed promotion. The goal is to test whether the failure is primarily checkpoint selection / selection bias before opening another PPO branch.

Governance:
- Baselines: canonical `risk_budget_usage_v1`; direct comparison `drawdown_rebound_reward_train_v1`.
- Role: `diagnose`.
- Changed layer: `selection` only. No new training and no shared-code behavior change.
- Target window: strict OOS plus saved walk-forward windows `wf_2020`, `wf_2022`, `wf_2023_2024`.
- Acceptance criteria: identify a retained checkpoint that reaches `relative_return >= +30pp` and `MaxDD <= 15%` on strict OOS and all saved walk-forward windows, or prove checkpoint selection is not the next useful lever.
- Expected trade-off: selecting lower-risk checkpoints may reduce rebound participation; that is acceptable only if walk-forward drawdown and excess remain stable.
- Failure mode: validation-selected checkpoints rank well in validation but below median or fail the hard target out of sample.
- Comparison-table row key: `drawdown_rebound_checkpoint_pbo_v1`.
- Notebook rule: this `.ipynb` is the primary single-run conclusion artifact.

Historical non-duplication:
- Closest success: `predictive_drawdown_risk_gate_v1` found a strict-OOS selector that passed `+30pp / <=15%`, but walk-forward blocked it.
- Closest failure: `drawdown_rebound_reward_train_v1` moved risk upstream into reward and still failed strict + walk-forward targets.
- Why this is not a duplicate: this does not alter reward or caps; it evaluates retained checkpoints and PBO-style selection behavior already produced by the reward run.

External basis:
- Deflated Sharpe Ratio / multiple-testing warning: https://www.davidhbailey.com/dhbpapers/deflated-sharpe.pdf
- Probability of Backtest Overfitting: https://doi.org/10.21314/JCF.2016.322
- Qlib GitHub research-to-production harness: https://github.com/microsoft/qlib
- Qlib paper: https://arxiv.org/abs/2009.11189


In [1]:
from __future__ import annotations

import json
import math
import os
import subprocess
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'AGENTS.md').exists() and (path / 'src' / 'soxl_quant').exists():
            return path
    raise RuntimeError(f'could not locate repo root from {start}')


ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ['PYTHONPATH'] = 'src:.'

CARD = 'drawdown_rebound_checkpoint_pbo_v1'
SOURCE_PROFILE = 'drawdown_rebound_reward_train_v1'
CARD_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / CARD
EXPORT_DIR = CARD_DIR / 'exports'
LOG_DIR = CARD_DIR / 'logs'
POOR_DIR = CARD_DIR / 'poor_windows'
MODEL_DIR = ROOT / 'runtime' / 'model' / 'card' / SOURCE_PROFILE
REWARD_CARD_DIR = ROOT / 'runtime' / 'reward' / 'card' / SOURCE_PROFILE
for path in [CARD_DIR, EXPORT_DIR, LOG_DIR, POOR_DIR]:
    path.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
WINDOWS = {
    'strict_oos': {'summary': MODEL_DIR / f'soxl_{SOURCE_PROFILE}_summary.json', 'test_start': '', 'test_end': ''},
    'wf_2020': {'summary': MODEL_DIR / f'soxl_{SOURCE_PROFILE}_wf_2020_summary.json', 'test_start': '2020-01-01', 'test_end': '2020-12-31'},
    'wf_2022': {'summary': MODEL_DIR / f'soxl_{SOURCE_PROFILE}_wf_2022_summary.json', 'test_start': '2022-01-01', 'test_end': '2022-12-31'},
    'wf_2023_2024': {'summary': MODEL_DIR / f'soxl_{SOURCE_PROFILE}_wf_2023_2024_summary.json', 'test_start': '2024-01-01', 'test_end': '2024-12-31'},
}

print(f'card_dir={CARD_DIR.relative_to(ROOT)}')
print(f'source_profile={SOURCE_PROFILE}')


card_dir=runtime/model_selection/card/drawdown_rebound_checkpoint_pbo_v1
source_profile=drawdown_rebound_reward_train_v1


In [2]:
def load_json(path: Path) -> dict[str, Any]:
    with path.open(encoding='utf-8') as f:
        return json.load(f)


def checkpoint_rows() -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for window, spec in WINDOWS.items():
        summary = load_json(spec['summary'])
        selected_path = Path(summary.get('model_path', ''))
        selected_iter = None
        selected_score = None
        if summary.get('selection_history'):
            best = max(summary['selection_history'], key=lambda row: float(row.get('score', float('-inf'))))
            selected_iter = int(best.get('iter'))
            selected_score = float(best.get('score'))
        for chk in summary.get('retained_checkpoints', []):
            metrics = chk.get('metrics', {}) or {}
            path = Path(chk['path'])
            rows.append({
                'window': window,
                'iter': int(chk['iter']),
                'validation_score': float(chk.get('score', float('nan'))),
                'validation_return': float(metrics.get('return', float('nan'))),
                'validation_sharpe': float(metrics.get('sharpe', float('nan'))),
                'validation_max_dd': float(metrics.get('max_dd', float('nan'))),
                'validation_avg_pos': float(metrics.get('avg_pos', float('nan'))),
                'actor_path': str(path),
                'actor_exists': path.exists(),
                'selected_by_summary': selected_iter == int(chk['iter']),
                'summary_selected_iter': selected_iter,
                'summary_selected_score': selected_score,
                'summary_model_path': str(selected_path),
                'test_start': spec['test_start'],
                'test_end': spec['test_end'],
            })
    return pd.DataFrame(rows)


checkpoints = checkpoint_rows()
checkpoints.to_csv(CARD_DIR / 'checkpoint_inventory.csv', index=False)
display(checkpoints[['window', 'iter', 'validation_score', 'actor_exists', 'selected_by_summary', 'test_start', 'test_end']])
if not checkpoints['actor_exists'].all():
    missing = checkpoints.loc[~checkpoints['actor_exists'], ['window', 'iter', 'actor_path']]
    print('missing checkpoint actors; existing exported CSVs will be reused if present')
    display(missing)


,window,iter,validation_score,actor_exists,selected_by_summary,test_start,test_end
0,strict_oos,0,-9.063366,True,False,,
1,strict_oos,4,14.289034,True,True,,
2,strict_oos,8,11.237874,True,False,,
3,wf_2020,0,-9.063366,True,False,2020-01-01,2020-12-31
4,wf_2020,4,14.289034,True,True,2020-01-01,2020-12-31
5,wf_2020,8,11.237874,True,False,2020-01-01,2020-12-31
6,wf_2022,0,29.148544,True,True,2022-01-01,2022-12-31
7,wf_2022,4,29.061986,True,False,2022-01-01,2022-12-31
8,wf_2022,8,13.470619,True,False,2022-01-01,2022-12-31
9,wf_2023_2024,0,67.750606,True,True,2024-01-01,2024-12-31


In [3]:
def export_checkpoint_daily(row: pd.Series) -> Path:
    window = str(row['window'])
    iteration = int(row['iter'])
    output = EXPORT_DIR / f'{window}_iter{iteration}_daily.csv'
    if output.exists() and output.stat().st_size > 0:
        return output
    actor_path = Path(str(row['actor_path']))
    if not actor_path.exists():
        raise FileNotFoundError(f'missing actor for {window} iter {iteration}: {actor_path}')
    cmd = [
        str(ROOT / 'qlib_venv' / 'bin' / 'python'),
        str(ROOT / 'scripts' / 'export_profile_daily_backtest.py'),
        '--profile', SOURCE_PROFILE,
        '--model', str(actor_path),
        '--output', str(output),
    ]
    if row['test_start'] and row['test_end']:
        cmd.extend(['--test-start', str(row['test_start']), '--test-end', str(row['test_end'])])
    env = {'PYTHONPATH': 'src:.', **dict(os.environ)}
    proc = subprocess.run(cmd, cwd=ROOT, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    (LOG_DIR / f'{window}_iter{iteration}.stdout.log').write_text(proc.stdout, encoding='utf-8')
    (LOG_DIR / f'{window}_iter{iteration}.stderr.log').write_text(proc.stderr, encoding='utf-8')
    if proc.returncode != 0:
        raise RuntimeError(f'export failed for {window} iter {iteration}: {proc.stderr[-2000:]}')
    return output


exports = []
for _, row in checkpoints.iterrows():
    path = export_checkpoint_daily(row)
    exports.append(path)
checkpoints['daily_csv'] = [str(path.relative_to(ROOT)) for path in exports]
checkpoints.to_csv(CARD_DIR / 'checkpoint_inventory.csv', index=False)
print(f'exported_or_reused={len(exports)}')


exported_or_reused=12


In [4]:
def pct_return(values: pd.Series) -> float:
    first = float(values.iloc[0])
    last = float(values.iloc[-1])
    return (last / first - 1.0) * 100.0 if first else float('nan')


def max_drawdown(values: pd.Series) -> float:
    arr = values.to_numpy(dtype=float)
    peak = np.maximum.accumulate(arr)
    dd = arr / peak - 1.0
    return abs(float(np.nanmin(dd)) * 100.0)


def sharpe_from_nav(values: pd.Series) -> float:
    rets = values.pct_change().dropna()
    if rets.empty or float(rets.std()) == 0.0:
        return 0.0
    return float(rets.mean() / (rets.std() + 1e-12) * np.sqrt(252))


def daily_metrics(path: Path) -> dict[str, Any]:
    df = pd.read_csv(path)
    df['datetime'] = pd.to_datetime(df['datetime'])
    trade_count = int((df['position'].diff().abs().fillna(0.0) > 0.001).sum())
    return {
        'rows': int(len(df)),
        'first_date': df['datetime'].min().date().isoformat(),
        'last_date': df['datetime'].max().date().isoformat(),
        'return': pct_return(df['strategy_nav']),
        'buyhold_return': pct_return(df['buyhold_nav']),
        'relative_return': pct_return(df['strategy_nav']) - pct_return(df['buyhold_nav']),
        'sharpe': sharpe_from_nav(df['strategy_nav']),
        'max_dd': max_drawdown(df['strategy_nav']),
        'avg_pos': float(df['position'].mean()),
        'trades': trade_count,
    }


metric_rows = []
for _, row in checkpoints.iterrows():
    out = row.to_dict()
    metrics = daily_metrics(ROOT / row['daily_csv'])
    out.update(metrics)
    out['target_pass'] = bool(metrics['relative_return'] >= TARGET_RELATIVE_RETURN and metrics['max_dd'] <= TARGET_MAX_DD)
    out['oos_score'] = (
        metrics['relative_return']
        - 4.0 * max(metrics['max_dd'] - TARGET_MAX_DD, 0.0)
        - 2.0 * max(TARGET_RELATIVE_RETURN - metrics['relative_return'], 0.0)
    )
    metric_rows.append(out)

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(CARD_DIR / 'checkpoint_metrics.csv', index=False)
display(metrics[['window', 'iter', 'selected_by_summary', 'validation_score', 'return', 'relative_return', 'max_dd', 'avg_pos', 'trades', 'target_pass', 'oos_score']].sort_values(['window', 'iter']))


,window,iter,selected_by_summary,validation_score,return,relative_return,max_dd,avg_pos,trades,target_pass,oos_score
0,strict_oos,0,False,-9.063366,603.967253,-212.545246,49.813996,0.369679,1414,False,-836.891725
1,strict_oos,4,True,14.289034,1394.143087,577.630587,56.302225,0.505075,1415,False,412.421686
2,strict_oos,8,False,11.237874,1426.815114,610.302614,54.550019,0.493176,1432,False,452.102538
3,wf_2020,0,False,-9.063366,81.467661,23.374091,15.518126,0.287147,173,False,8.049767
4,wf_2020,4,True,14.289034,111.640959,53.547389,22.456890,0.401614,173,False,23.719830
5,wf_2020,8,False,11.237874,112.525408,54.431837,23.801426,0.396220,176,False,19.226134
6,wf_2022,0,True,29.148544,-21.140962,65.447110,30.816084,0.196932,181,False,2.182773
7,wf_2022,4,False,29.061986,-32.858953,53.729119,39.503343,0.242966,183,False,-44.284253
8,wf_2022,8,False,13.470619,-21.006847,65.581224,30.715739,0.197470,181,False,2.718267
9,wf_2023_2024,0,True,67.750606,-16.171093,-15.814458,31.455751,0.277489,175,False,-173.266377


In [5]:
# Replay parity audit: committed summary daily vs current actor replay.
committed_daily = MODEL_DIR / f'{SOURCE_PROFILE}_soxl_daily.csv'
summary_actor_daily = EXPORT_DIR / 'strict_oos_summary_actor_daily.csv'
summary_actor_path = Path(load_json(WINDOWS['strict_oos']['summary'])['model_path'])
if not summary_actor_daily.exists() and summary_actor_path.exists():
    cmd = [
        str(ROOT / 'qlib_venv' / 'bin' / 'python'),
        str(ROOT / 'scripts' / 'export_profile_daily_backtest.py'),
        '--profile', SOURCE_PROFILE,
        '--model', str(summary_actor_path),
        '--output', str(summary_actor_daily),
    ]
    proc = subprocess.run(cmd, cwd=ROOT, env={'PYTHONPATH': 'src:.', **dict(os.environ)}, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    (LOG_DIR / 'strict_oos_summary_actor.stdout.log').write_text(proc.stdout, encoding='utf-8')
    (LOG_DIR / 'strict_oos_summary_actor.stderr.log').write_text(proc.stderr, encoding='utf-8')
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr[-2000:])

parity_rows = []
if committed_daily.exists():
    committed_metrics = daily_metrics(committed_daily)
    parity_rows.append({'source': 'committed_summary_daily', **committed_metrics, 'daily_csv': str(committed_daily.relative_to(ROOT))})
if summary_actor_daily.exists():
    replay_metrics = daily_metrics(summary_actor_daily)
    parity_rows.append({'source': 'current_summary_actor_replay', **replay_metrics, 'daily_csv': str(summary_actor_daily.relative_to(ROOT))})
if not metrics.empty:
    selected_metric = metrics[(metrics['window'] == 'strict_oos') & (metrics['selected_by_summary'])].iloc[0]
    parity_rows.append({'source': 'selected_checkpoint_replay', **{k: selected_metric[k] for k in ['rows', 'first_date', 'last_date', 'return', 'buyhold_return', 'relative_return', 'sharpe', 'max_dd', 'avg_pos', 'trades']}, 'daily_csv': selected_metric['daily_csv']})

parity = pd.DataFrame(parity_rows)
if not parity.empty:
    base = parity.iloc[0]
    parity['return_delta_vs_first'] = parity['return'] - float(base['return'])
    parity['max_dd_delta_vs_first'] = parity['max_dd'] - float(base['max_dd'])
    parity['avg_pos_delta_vs_first'] = parity['avg_pos'] - float(base['avg_pos'])
    parity['parity_with_first'] = (
        parity['return_delta_vs_first'].abs() < 1e-6
    ) & (
        parity['max_dd_delta_vs_first'].abs() < 1e-6
    ) & (
        parity['avg_pos_delta_vs_first'].abs() < 1e-6
    )
parity.to_csv(CARD_DIR / 'summary_daily_parity.csv', index=False)
display(parity)
parity_mismatch = bool((not parity.empty) and not parity['parity_with_first'].fillna(False).all())
print('parity_mismatch', parity_mismatch)


,source,rows,first_date,last_date,return,buyhold_return,relative_return,sharpe,max_dd,avg_pos,trades,daily_csv,return_delta_vs_first,max_dd_delta_vs_first,avg_pos_delta_vs_first,parity_with_first
0,committed_summary_daily,1605,2020-01-02,2026-05-21,392.580698,816.5125,-423.931802,1.026549,37.676486,0.268595,1392,runtime/model/card/drawdown_rebound_reward_tra...,0.000000,0.00000,0.00000,True
1,current_summary_actor_replay,1605,2020-01-02,2026-05-21,1394.143087,816.5125,577.630587,1.073856,56.302225,0.505075,1415,runtime/model_selection/card/drawdown_rebound_...,1001.562389,18.62574,0.23648,False
2,selected_checkpoint_replay,1605,2020-01-02,2026-05-21,1394.143087,816.5125,577.630587,1.073856,56.302225,0.505075,1415,runtime/model_selection/card/drawdown_rebound_...,1001.562389,18.62574,0.23648,False


parity_mismatch True


In [6]:
selection_rows = []
for window, group in metrics.groupby('window'):
    ranked = group.sort_values('oos_score', ascending=False).reset_index(drop=True)
    selected = group[group['selected_by_summary']].copy()
    if selected.empty:
        selected = group[group['iter'] == group['summary_selected_iter'].iloc[0]].copy()
    selected_row = selected.iloc[0]
    median_score = float(group['oos_score'].median())
    best_row = ranked.iloc[0]
    selection_rows.append({
        'window': window,
        'summary_selected_iter': int(selected_row['iter']),
        'summary_selected_validation_score': float(selected_row['validation_score']),
        'summary_selected_oos_score': float(selected_row['oos_score']),
        'summary_selected_target_pass': bool(selected_row['target_pass']),
        'summary_selected_relative_return': float(selected_row['relative_return']),
        'summary_selected_max_dd': float(selected_row['max_dd']),
        'best_oos_iter': int(best_row['iter']),
        'best_oos_score': float(best_row['oos_score']),
        'best_oos_target_pass': bool(best_row['target_pass']),
        'best_oos_relative_return': float(best_row['relative_return']),
        'best_oos_max_dd': float(best_row['max_dd']),
        'selected_below_median_oos': bool(selected_row['oos_score'] < median_score),
        'selected_not_best_oos': bool(int(selected_row['iter']) != int(best_row['iter'])),
        'oos_score_gap_to_best': float(best_row['oos_score'] - selected_row['oos_score']),
    })
selection_table = pd.DataFrame(selection_rows).sort_values('window')
selection_table.to_csv(CARD_DIR / 'checkpoint_selection_table.csv', index=False)

window_pass_count = int(selection_table['summary_selected_target_pass'].sum())
best_pass_count = int(selection_table['best_oos_target_pass'].sum())
pbo_proxy_below_median_rate = float(selection_table['selected_below_median_oos'].mean())
pbo_proxy_not_best_rate = float(selection_table['selected_not_best_oos'].mean())

display(selection_table)
print('summary_selected_pass_count', window_pass_count, 'best_oos_pass_count', best_pass_count)
print('pbo_proxy_below_median_rate', pbo_proxy_below_median_rate, 'pbo_proxy_not_best_rate', pbo_proxy_not_best_rate)


,window,summary_selected_iter,summary_selected_validation_score,summary_selected_oos_score,summary_selected_target_pass,summary_selected_relative_return,summary_selected_max_dd,best_oos_iter,best_oos_score,best_oos_target_pass,best_oos_relative_return,best_oos_max_dd,selected_below_median_oos,selected_not_best_oos,oos_score_gap_to_best
0,strict_oos,4,14.289034,412.421686,False,577.630587,56.302225,8,452.102538,False,610.302614,54.550019,False,True,39.680852
1,wf_2020,4,14.289034,23.719830,False,53.547389,22.456890,4,23.719830,False,53.547389,22.456890,False,False,0.000000
2,wf_2022,0,29.148544,2.182773,False,65.447110,30.816084,8,2.718267,False,65.581224,30.715739,False,True,0.535494
3,wf_2023_2024,0,67.750606,-173.266377,False,-15.814458,31.455751,0,-173.266377,False,-15.814458,31.455751,False,False,0.000000


summary_selected_pass_count 0 best_oos_pass_count 0
pbo_proxy_below_median_rate 0.0 pbo_proxy_not_best_rate 0.5


In [7]:
from src.soxl_quant.app.oos_diagnostics import run_oos_diagnostic

EXTERNAL_REFS = [
    {'label': 'Deflated Sharpe Ratio', 'url': 'https://www.davidhbailey.com/dhbpapers/deflated-sharpe.pdf', 'note': 'selection-bias warning for repeated trials'},
    {'label': 'Probability of Backtest Overfitting', 'url': 'https://doi.org/10.21314/JCF.2016.322', 'note': 'strategy-selection overfit diagnostic'},
    {'label': 'Qlib GitHub', 'url': 'https://github.com/microsoft/qlib', 'note': 'repo-owned qlib workflow'},
]

strict_metrics = metrics[metrics['window'] == 'strict_oos'].copy()
selected_strict = strict_metrics[strict_metrics['selected_by_summary']].iloc[0]
best_low_dd = strict_metrics.sort_values(['max_dd', 'relative_return'], ascending=[True, False]).iloc[0]
poor_inputs = [
    ('summary_selected_strict', ROOT / selected_strict['daily_csv']),
    (f'low_dd_iter{int(best_low_dd["iter"])}', ROOT / best_low_dd['daily_csv']),
]
if poor_inputs[1][1] == poor_inputs[0][1]:
    poor_inputs = poor_inputs[:1]

poor_manifests = []
for label, daily_path in poor_inputs:
    out_dir = POOR_DIR / label
    manifest = run_oos_diagnostic(
        daily_csv=daily_path,
        profile=f'{SOURCE_PROFILE}:{label}',
        output_dir=out_dir,
        oos_start='2020-01-02',
        window_size=63,
        top_windows=10,
        top_features=20,
        target_excess=TARGET_RELATIVE_RETURN,
        target_max_dd=TARGET_MAX_DD,
        external_references=EXTERNAL_REFS,
    )
    poor_manifests.append(manifest)

for manifest in poor_manifests:
    print(manifest['profile'], 'windows', manifest['windows'], 'features', manifest['features'], 'dir', manifest['artifacts']['review'])

poor_summary_rows = []
for manifest in poor_manifests:
    windows = pd.read_csv(manifest['artifacts']['windows'])
    features = pd.read_csv(manifest['artifacts']['features'])
    top_window = windows.iloc[0].to_dict() if not windows.empty else {}
    top_features = features.head(8)['feature'].tolist() if not features.empty else []
    poor_summary_rows.append({
        'profile': manifest['profile'],
        'top_window_start': top_window.get('start'),
        'top_window_end': top_window.get('end'),
        'top_window_excess_return': top_window.get('excess_return'),
        'top_window_max_dd': top_window.get('max_dd'),
        'top_feature_1': top_features[0] if len(top_features) > 0 else None,
        'top_feature_2': top_features[1] if len(top_features) > 1 else None,
        'top_feature_3': top_features[2] if len(top_features) > 2 else None,
    })
poor_summary = pd.DataFrame(poor_summary_rows)
poor_summary.to_csv(CARD_DIR / 'poor_window_summary.csv', index=False)
display(poor_summary)


[58463:MainThread](2026-05-25 08:55:55,068) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[58463:MainThread](2026-05-25 08:55:55,743) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[58463:MainThread](2026-05-25 08:55:55,745) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[58463:MainThread](2026-05-25 08:55:58,394) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[58463:MainThread](2026-05-25 08:55:58,397) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


...

[58463:MainThread](2026-05-25 08:56:01,475) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[58463:MainThread](2026-05-25 08:56:01,475) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[58463:MainThread](2026-05-25 08:56:01,476) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Init data Done


drawdown_rebound_reward_train_v1:summary_selected_strict windows 10 features 20 dir /Users/rziiiii/.openclaw/workspace/runtime/model_selection/card/drawdown_rebound_checkpoint_pbo_v1/poor_windows/summary_selected_strict/oos_diagnostic_review.md
drawdown_rebound_reward_train_v1:low_dd_iter0 windows 10 features 20 dir /Users/rziiiii/.openclaw/workspace/runtime/model_selection/card/drawdown_rebound_checkpoint_pbo_v1/poor_windows/low_dd_iter0/oos_diagnostic_review.md


,profile,top_window_start,top_window_end,top_window_excess_return,top_window_max_dd,top_feature_1,top_feature_2,top_feature_3
0,drawdown_rebound_reward_train_v1:summary_selec...,2025-04-08,2025-07-09,-166.357595,12.397085,vol_of_vol_21,tracking_error_vol_21,decay_63
1,drawdown_rebound_reward_train_v1:low_dd_iter0,2025-04-08,2025-07-09,-183.228387,11.311921,decay_63,vol_of_vol_21,tracking_error_vol_21


In [8]:
def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [json_safe(v) for v in value]
    if hasattr(value, 'item') and not isinstance(value, (str, bytes, bool)):
        try:
            value = value.item()
        except (TypeError, ValueError):
            pass
    if value is None:
        return None
    if isinstance(value, float) and (math.isnan(value) or math.isinf(value)):
        return None
    if isinstance(value, (str, int, float, bool)):
        return value
    return str(value)

any_checkpoint_full_pass = bool(metrics.groupby('iter')['target_pass'].sum().max() == len(WINDOWS))
any_window_target_gaps = selection_table[['window', 'best_oos_iter', 'best_oos_target_pass', 'best_oos_relative_return', 'best_oos_max_dd']].to_dict(orient='records')

if 'parity_mismatch' in globals() and parity_mismatch:
    decision = 'diagnostic_replay_parity_mismatch_no_promotion'
elif any_checkpoint_full_pass:
    decision = 'candidate_checkpoint_requires_confirmation'
elif best_pass_count == 0:
    decision = 'diagnostic_no_checkpoint_window_pass'
elif pbo_proxy_below_median_rate >= 0.5 or pbo_proxy_not_best_rate >= 0.5:
    decision = 'diagnostic_checkpoint_selection_bias_detected'
else:
    decision = 'diagnostic_checkpoint_selection_not_sufficient'

interpretation = []
if 'parity_mismatch' in globals() and parity_mismatch:
    interpretation.append('Committed summary daily and current actor replay are not numerically identical; this blocks using local checkpoint replay as promotion proof.')
if best_pass_count < len(WINDOWS):
    interpretation.append(f'Even oracle OOS checkpoint choice passes only {best_pass_count}/{len(WINDOWS)} windows, so checkpoint selection alone cannot satisfy the target.')
if window_pass_count < len(WINDOWS):
    interpretation.append(f'The summary-selected checkpoints pass only {window_pass_count}/{len(WINDOWS)} windows under +30pp / <=15% criteria.')
if pbo_proxy_below_median_rate > 0:
    interpretation.append(f'Validation-selected checkpoints are below median OOS score in {pbo_proxy_below_median_rate:.0%} of windows, consistent with selection-bias risk.')
if not poor_summary.empty:
    first = poor_summary.iloc[0]
    interpretation.append(f'Worst strict poor window starts {first["top_window_start"]} and is associated with {first["top_feature_1"]}, {first["top_feature_2"]}, {first["top_feature_3"]}.')
interpretation.append('Next useful lever should not be longer training of this reward branch; use poor-window features to design a regime-specific actor/composer change, then validate with saved walk-forward.')

selection = {
    'experiment': CARD,
    'source_profile': SOURCE_PROFILE,
    'decision': decision,
    'target': {'relative_return_min_pp': TARGET_RELATIVE_RETURN, 'max_dd_max_pct': TARGET_MAX_DD},
    'summary_selected_pass_count': window_pass_count,
    'best_oos_pass_count': best_pass_count,
    'pbo_proxy_below_median_rate': pbo_proxy_below_median_rate,
    'pbo_proxy_not_best_rate': pbo_proxy_not_best_rate,
    'parity_mismatch': bool(globals().get('parity_mismatch', False)),
    'window_best_rows': any_window_target_gaps,
    'poor_window_summary': poor_summary.to_dict(orient='records'),
    'interpretation': interpretation,
    'artifacts': {
        'notebook': str((CARD_DIR / f'{CARD}.ipynb').relative_to(ROOT)),
        'checkpoint_inventory': str((CARD_DIR / 'checkpoint_inventory.csv').relative_to(ROOT)),
        'checkpoint_metrics': str((CARD_DIR / 'checkpoint_metrics.csv').relative_to(ROOT)),
        'checkpoint_selection_table': str((CARD_DIR / 'checkpoint_selection_table.csv').relative_to(ROOT)),
        'poor_window_summary': str((CARD_DIR / 'poor_window_summary.csv').relative_to(ROOT)),
        'summary_daily_parity': str((CARD_DIR / 'summary_daily_parity.csv').relative_to(ROOT)),
    },
}
(CARD_DIR / 'selection.json').write_text(json.dumps(json_safe(selection), indent=2, allow_nan=False), encoding='utf-8')

lines = [
    f'# {CARD} conclusion',
    '',
    f'- Decision: `{decision}`',
    f'- Summary-selected target pass count: `{window_pass_count}/{len(WINDOWS)}`',
    f'- Oracle OOS checkpoint target pass count: `{best_pass_count}/{len(WINDOWS)}`',
    f'- PBO proxy below-median rate: `{pbo_proxy_below_median_rate:.0%}`',
    f'- PBO proxy not-best rate: `{pbo_proxy_not_best_rate:.0%}`',
    f'- Replay parity mismatch: `{str(bool(globals().get("parity_mismatch", False))).lower()}`',
    '',
    '## Window Best Rows',
    '',
    '| window | best_iter | target_pass | relative | MaxDD |',
    '| --- | ---: | --- | ---: | ---: |',
]
for row in any_window_target_gaps:
    lines.append(f"| `{row['window']}` | `{int(row['best_oos_iter'])}` | `{str(bool(row['best_oos_target_pass'])).lower()}` | `{float(row['best_oos_relative_return']):.2f}pp` | `{float(row['best_oos_max_dd']):.2f}%` |")
lines.extend(['', '## Interpretation'])
lines.extend(f'- {item}' for item in interpretation)
conclusion_md = '\n'.join(lines)
display(Markdown(conclusion_md))


# drawdown_rebound_checkpoint_pbo_v1 conclusion

- Decision: `diagnostic_replay_parity_mismatch_no_promotion`
- Summary-selected target pass count: `0/4`
- Oracle OOS checkpoint target pass count: `0/4`
- PBO proxy below-median rate: `0%`
- PBO proxy not-best rate: `50%`
- Replay parity mismatch: `true`

## Window Best Rows

| window | best_iter | target_pass | relative | MaxDD |
| --- | ---: | --- | ---: | ---: |
| `strict_oos` | `8` | `false` | `610.30pp` | `54.55%` |
| `wf_2020` | `4` | `false` | `53.55pp` | `22.46%` |
| `wf_2022` | `8` | `false` | `65.58pp` | `30.72%` |
| `wf_2023_2024` | `0` | `false` | `-15.81pp` | `31.46%` |

## Interpretation
- Committed summary daily and current actor replay are not numerically identical; this blocks using local checkpoint replay as promotion proof.
- Even oracle OOS checkpoint choice passes only 0/4 windows, so checkpoint selection alone cannot satisfy the target.
- The summary-selected checkpoints pass only 0/4 windows under +30pp / <=15% criteria.
- Worst strict poor window starts 2025-04-08 and is associated with vol_of_vol_21, tracking_error_vol_21, decay_63.
- Next useful lever should not be longer training of this reward branch; use poor-window features to design a regime-specific actor/composer change, then validate with saved walk-forward.